# Budgerigar：FSDD 单发音数据
在 Colab 下载、固定版本、生成 manifest 并审计短数字发音。

In [ ]:
#@title 1. 更新项目
REPO_DIR='/content/Budgerigar'
from pathlib import Path
import subprocess,sys,importlib,shutil,datetime
repo=Path(REPO_DIR)
if not (repo/'.git').is_dir():subprocess.run(['git','clone','--depth=1','https://github.com/DoctorAwe/Budgerigar.git',REPO_DIR],check=True)
else:
 pull=subprocess.run(['git','-C',REPO_DIR,'pull','--ff-only'],text=True,capture_output=True);print(pull.stdout,pull.stderr)
 if pull.returncode:
  backup=repo.with_name(f'Budgerigar_backup_{datetime.datetime.now():%H%M%S}');shutil.move(str(repo),str(backup));subprocess.run(['git','clone','--depth=1','https://github.com/DoctorAwe/Budgerigar.git',REPO_DIR],check=True)
subprocess.run([sys.executable,'-m','pip','install','-q','-e',f'{REPO_DIR}[train]'],check=True);sys.path.insert(0,REPO_DIR)
for name in [k for k in list(sys.modules) if k=='budgerigar' or k.startswith('budgerigar.')]:del sys.modules[name]
importlib.invalidate_caches();print(subprocess.run(['git','-C',REPO_DIR,'rev-parse','--short','HEAD'],capture_output=True,text=True).stdout.strip())

In [ ]:
#@title 2. 下载并生成 manifest
from google.colab import drive
drive.mount('/content/drive');WORK_ROOT=Path('/content/drive/MyDrive/Budgerigar');FSDD_ROOT=WORK_ROOT/'data'/'free-spoken-digit-dataset'
from budgerigar.fsdd_data import prepare_fsdd,build_fsdd_manifest,write_fsdd_manifest
REVISION='master' #@param {type:'string'}
commit=prepare_fsdd(FSDD_ROOT,REVISION);rows=build_fsdd_manifest(FSDD_ROOT);MANIFEST=WORK_ROOT/'manifests'/'fsdd.jsonl';report=write_fsdd_manifest(rows,MANIFEST,commit)
import json;print(json.dumps(report,ensure_ascii=False,indent=2))

In [ ]:
#@title 3. 音频与标签审计
from collections import Counter
assert report['records']==3000 and len(report['speakers'])==6
assert Counter(row['label'] for row in rows)=={i:300 for i in range(10)}
assert all(row['channels']==1 and row['sample_width']==2 for row in rows)
print('sample rates:',Counter(row['sample_rate'] for row in rows));print('duration range:',min(x['duration'] for x in rows),max(x['duration'] for x in rows));print('manifest:',MANIFEST)